In [69]:
import pandas as pd
import glob

# Define a list to hold all DataFrames
all_gpu_data = []

# Pattern to find all current and historical GPU pricing CSV files
# This includes newly fetched data (gpu_prices_YYYYMMDD_HHMMSS.csv)
# and historical data (gpu_pricing_history.csv if it exists)
csv_patterns = ["gpu_prices_*.csv", "gpu_pricing_history.csv"]

print("Searching for GPU pricing CSV files...")

for pattern in csv_patterns:
    for file_path in glob.glob(pattern):
        try:
            df_temp = pd.read_csv(file_path)
            all_gpu_data.append(df_temp)
            print(f"Loaded {len(df_temp)} records from {file_path}")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

if all_gpu_data:
    # Concatenate all DataFrames into a single DataFrame
    combined_df = pd.concat(all_gpu_data, ignore_index=True)

    # --- New: Save the unfiltered combined_df ---
    combined_csv_filename_unfiltered = "combined_gpu_data_unfiltered.csv"
    combined_df.to_csv(combined_csv_filename_unfiltered, index=False)
    print(f"Unfiltered combined data saved to '{combined_csv_filename_unfiltered}'. Total records: {len(combined_df)}.")
    # ------------------------------------------

    # Remove potential duplicate rows across files
    # It's important to define a subset of columns that uniquely identify a record
    # If 'extracted_at' is different for the same GPU configuration at different times,
    # then dropping based on all columns might be too aggressive.
    # For this, let's assume `gpu`, `provider`, `vram_gb`, `price_per_hour_usd`, `pricing_type`
    # are key identifiers. We keep the latest entry for duplicates if `last_updated` exists.
    if 'last_updated' in combined_df.columns:
        # Ensure 'last_updated' is datetime for proper sorting
        combined_df['last_updated'] = pd.to_datetime(combined_df['last_updated'], errors='coerce', utc=True)
        # Sort by last_updated and drop duplicates, keeping the most recent
        combined_df = combined_df.sort_values(by=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type', 'last_updated'], ascending=False)
        combined_df.drop_duplicates(subset=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type'], keep='first', inplace=True)
    else:
        # If no last_updated, just drop duplicates based on core identifying columns
        combined_df.drop_duplicates(subset=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type'], keep='first', inplace=True)

    # Define the output CSV file name
    combined_csv_filename = "combined_gpu_data.csv"

    # Save the combined DataFrame to a CSV file
    combined_df.to_csv(combined_csv_filename, index=False)

    print(f"\nSuccessfully combined all GPU data from {len(all_gpu_data)} files.")
    print(f"Total unique records in combined DataFrame: {len(combined_df)}.")
    print(f"Combined data saved to '{combined_csv_filename}'.")

    # Display the first few rows of the combined DataFrame
    print("\nFirst 5 rows of the combined data:")
    display(combined_df.head())
else:
    print("No GPU pricing CSV files found to combine.")

Searching for GPU pricing CSV files...
Loaded 1042 records from gpu_prices_20260710_002006.csv
Loaded 1042 records from gpu_prices_20260710_003104.csv
Loaded 1047 records from gpu_prices_20260710_171043.csv
Loaded 1033 records from gpu_prices_20260711_175520.csv
Loaded 1033 records from gpu_prices_20260711_193154.csv
Loaded 1031 records from gpu_prices_20260712_201314.csv
Loaded 1046 records from gpu_prices_20260713_165406.csv
Loaded 1046 records from gpu_prices_20260713_213739.csv
Loaded 1051 records from gpu_prices_20260714_163003.csv
Loaded 965 records from gpu_prices_20260714_233138.csv
Loaded 1010 records from gpu_prices_20260715_211606.csv
Unfiltered combined data saved to 'combined_gpu_data_unfiltered.csv'. Total records: 11346.

Successfully combined all GPU data from 11 files.
Total unique records in combined DataFrame: 879.
Combined data saved to 'combined_gpu_data.csv'.

First 5 rows of the combined data:


,provider,provider_slug,provider_url,gpu,gpu_slug,vram_gb,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,pricing_type,commitment_months,currency,exchange_rate_to_usd,source_url,last_updated,gpu_model,extracted_at
11333,Verda,verda,https://verda.com,Tesla V100,v100,32,Volta,8,8,0.1700,1.3600,on_demand,NaN,USD,1.0,https://api.verda.com/v1/docs#tag/instance-typ...,2026-07-15 00:01:28.642222+00:00,TESLA V100,20260715_211606
11337,Verda,verda,https://verda.com,Tesla V100,v100,32,Volta,8,8,0.0595,0.4760,spot,NaN,USD,1.0,https://api.verda.com/v1/docs#tag/instance-typ...,2026-07-15 00:01:28.663293+00:00,TESLA V100,20260715_211606
10655,Vast.ai,vast,https://vast.ai/,Tesla V100,v100,32,Volta,1,2,0.2689,0.2689,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-07-15 00:00:39.029572+00:00,TESLA V100,20260715_211606
6543,Vast.ai,vast,https://vast.ai/,Tesla V100,v100,32,Volta,1,2,0.2262,0.2262,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-07-13 00:00:25.890921+00:00,TESLA V100,20260713_165406
293,Vast.ai,vast,https://vast.ai/,Tesla V100,v100,32,Volta,2,2,0.1241,0.2482,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-07-02 00:00:12.602327+00:00,TESLA V100,20260710_002006


In [70]:
len(combined_df)

879

In [71]:
combined_df['gpu'].unique()
len(combined_df['gpu'].unique())

55

In [72]:
combined_df['provider'].unique()
len(combined_df['provider'].unique())

49

In [73]:
combined_df['gpu_slug'].unique()
len(combined_df['gpu_slug'].unique())

55

In [74]:
combined_df['last_updated'].unique()
len(combined_df['last_updated'].unique())

879

In [75]:
combined_df['price_per_hour_usd'].describe()

count    879.000000
mean       1.987419
std        2.311832
min        0.040000
25%        0.382500
50%        1.210000
75%        2.720550
max       18.000000
Name: price_per_hour_usd, dtype: float64

In [76]:
combined_df.groupby(
    ["gpu_model", "provider"]
)["price_per_hour_usd"].nunique().describe()

count    422.000000
mean       2.068720
std        2.007709
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       15.000000
Name: price_per_hour_usd, dtype: float64

In [77]:
combined_df.columns

Index(['provider', 'provider_slug', 'provider_url', 'gpu', 'gpu_slug',
       'vram_gb', 'architecture', 'gpu_count', 'max_gpus_per_node',
       'price_per_hour_usd', 'total_hourly_usd', 'pricing_type',
       'commitment_months', 'currency', 'exchange_rate_to_usd', 'source_url',
       'last_updated', 'gpu_model', 'extracted_at'],
      dtype='object')

In [78]:
import pandas as pd
import numpy as np
df=pd.read_csv("combined_gpu_data_unfiltered.csv")
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)

(11346, 19)
provider                   0
provider_slug              0
provider_url               0
gpu                        0
gpu_slug                   0
vram_gb                    0
architecture               0
gpu_count                  0
max_gpus_per_node          0
price_per_hour_usd         0
total_hourly_usd           0
pricing_type               0
commitment_months       9634
currency                   0
exchange_rate_to_usd       0
source_url                 0
last_updated               0
gpu_model                  0
extracted_at               0
dtype: int64
provider                 object
provider_slug            object
provider_url             object
gpu                      object
gpu_slug                 object
vram_gb                   int64
architecture             object
gpu_count                 int64
max_gpus_per_node         int64
price_per_hour_usd      float64
total_hourly_usd        float64
pricing_type             object
commitment_months       float64
currency